# Лабораторная работа: KNN для классификации и регрессии

**Цель:** построить воспроизводимые базовые и улучшенные модели sklearn, самостоятельно реализовать метод k ближайших соседей и сравнить четыре варианта на каждой задаче.

Порядок соответствует пунктам задания: 1 — условия и метрики; 2 — бейзлайн; 3 — гипотезы и улучшение; 4 — собственный алгоритм и сравнение. Перед каждой ячейкой кода есть пояснение, внутри — комментарии. Все таблицы и выводы ниже вычисляются на настоящих CSV из указанных источников.

## 1. Выбор начальных условий

### 1a. Классификация
[What Makes a Steam Game a Hit? 8,000 Games — vedauv](https://www.kaggle.com/datasets/vedauv/what-makes-a-steam-game-a-hit-8000-games).

Практический прототип: издатель до релиза оценивает вероятность коммерческого успеха, чтобы распределять маркетинговый бюджет. Цель `hit` уже дана автором: 1 для верхних 20% игр по выручке. **Данные синтетические**, что прямо указано в README и datasheet автора (копии в `data/steam/`). Это учебная модель реальной задачи, а не свидетельство закономерностей рынка. Порог определён автором на всём синтетическом наборе; мы не пересчитываем его на тесте. Уникальность набора среди студентов необходимо согласовать с преподавателем.

Для сценария до релиза исключены `revenue_musd`, `owners_k`, `num_reviews`, `peak_concurrent_players`; также исключены отзывы `steam_positive_pct`, `metacritic_score` и количество DLC `dlc_count`, для которых нет гарантии доступности до релиза. `game_id` и случайное название не являются содержательными признаками. `wishlists_k` автор описывает как дорелизные; просмотры трейлера и остальные характеристики считаем зафиксированными до релиза. Это допущение о моменте измерения, которое для реального внедрения необходимо проверять.

### 1b. Регрессия
[Video Games Sales — ulrikthygepedersen](https://www.kaggle.com/datasets/ulrikthygepedersen/video-games-sales).

Практический прототип: оценка мировых продаж игры по платформе, жанру, году и издателю для планирования тиража. Цель `global_sales` измеряется в **миллионах копий**. Убираем региональные продажи, сумма которых почти равна цели, и `rank`, вычисленный из продаж. Название используется только для группировки при разбиении, а не как признак. Продажи накоплены к моменту сбора данных, поэтому это оценка внутри исторической выборки, а не прогноз фиксированного горизонта. Данные о непродававшихся играх отсутствуют; результат нельзя автоматически переносить на весь рынок.

### 1c. Метрики
- Классификация: **Average Precision (AP)** — основная метрика выбора, оценивает ранжирование редкого положительного класса по всем порогам. У случайного ранжирования ориентир — доля положительных. Дополнительно F1, precision, recall для `hit=1`, balanced accuracy и ROC AUC. Accuracy приведена для наглядности, но при дисбалансе недостаточна. Порог фиксирован: вероятность > 0.5 (при равенстве выбирается класс 0).
- Регрессия: **MAE** — основная метрика, интерпретируется как средняя ошибка в миллионах копий; RMSE сильнее штрафует ошибки на крупных релизах; R² сравнивает с предсказанием среднего. Отрицательный R² допустим. Все метрики считаем в исходных единицах, даже после логарифмирования цели.

### Протокол эксперимента
20% данных оставляем на тест. Steam — стратифицированное случайное разбиение независимых синтетических строк. Продажи — разделение по нормализованному названию игры, чтобы версии одной игры не пересекались между частями. Внутри train используем 3-fold CV (также по группам для продаж). Заполнение пропусков, кодирование и масштабирование обучаются **внутри Pipeline на каждом train-fold**. Тест не участвует в выборе гипотез и параметров. Это оценка на новых названиях из той же исторической совокупности; для прогноза будущих лет нужен отдельный временной эксперимент.

Зависимости устанавливаются командой `python -m pip install -r requirements.txt`. Запускать ноутбук следует из корня проекта; CSV включены в проект, сеть при выполнении не нужна.

### Импорт и воспроизводимость
Фиксируем seed, ограничиваем число потоков, выводим версии библиотек. Параллелизм поиска ограничен, чтобы не перегружать память.

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "2")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "2")
from pathlib import Path
import hashlib
import json
import platform
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from IPython.display import display, Markdown
from sklearn.base import BaseEstimator, ClassifierMixin, RegressorMixin, TransformerMixin, clone
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import (train_test_split, GroupShuffleSplit, StratifiedKFold,
                                     GroupKFold, cross_val_score, GridSearchCV)
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.metrics import (average_precision_score, roc_auc_score, accuracy_score,
    balanced_accuracy_score, precision_score, recall_score, f1_score,
    mean_absolute_error, mean_squared_error, r2_score,
    ConfusionMatrixDisplay, PrecisionRecallDisplay)
from sklearn.utils.validation import check_X_y, check_array, check_is_fitted
from threadpoolctl import threadpool_limits
SEED = 42
np.random.seed(SEED)
thread_limit = threadpool_limits(limits=2)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 100)
ROOT = Path.cwd()
assert (ROOT / "data").exists(), "Откройте ноутбук из корня проекта"
RESULTS = ROOT / "results"
RESULTS.mkdir(exist_ok=True)
print({"Python": platform.python_version(), "numpy": np.__version__,
       "pandas": pd.__version__, "sklearn": sklearn.__version__})

### Загрузка и проверка исходных данных
Показываем размер, первые строки, пропуски и число дубликатов. SHA-256 позволяет установить, на каких именно версиях CSV получены результаты. Обязательные столбцы проверяются явно.

In [ ]:
steam_path = ROOT / "data/steam/steam_game_hits.csv"
sales_path = ROOT / "data/sales/video_games_sales.csv"
steam = pd.read_csv(steam_path)
sales = pd.read_csv(sales_path)
manifest = {}
for label, path, frame in [("Steam", steam_path, steam), ("Sales", sales_path, sales)]:
    manifest[label] = {"file": str(path.relative_to(ROOT)), "rows": len(frame),
                       "sha256": hashlib.sha256(path.read_bytes()).hexdigest()}
    print(label, frame.shape, "Полных дубликатов:", frame.duplicated().sum())
    display(frame.head(3))
    display(frame.isna().sum().rename("Пропуски").to_frame().T)
assert {"hit", "game_id", "wishlists_k", "revenue_musd"} <= set(steam)
assert {"name", "platform", "year", "genre", "publisher", "global_sales"} <= set(sales)
assert set(steam.hit.unique()) == {0, 1}
assert steam.game_id.is_unique
(RESULTS / "data_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")

### Подготовка признаков и единственное разбиение train/test
Удаляем полные дубликаты и строки без корректной цели. Строки без названия в продажах исключаем, потому что нельзя установить группу. Пропуски в признаках сохраняем для обработки внутри CV. Используем белый список признаков, чтобы случайно не добавить утечку.

In [ ]:
steam = steam.drop_duplicates().dropna(subset=["hit"]).reset_index(drop=True)
sales = sales.drop_duplicates().dropna(subset=["global_sales", "name"]).copy()
sales = sales.loc[np.isfinite(sales.global_sales) & (sales.global_sales >= 0)].reset_index(drop=True)
steam_num = ["release_year", "is_indie", "team_size", "price_usd", "has_multiplayer",
             "early_access", "controller_support", "languages_supported", "file_size_gb",
             "wishlists_k", "trailer_views_k"]
steam_cat = ["genre"]
sales_num, sales_cat = ["year"], ["platform", "genre", "publisher"]
Xc, yc = steam[steam_num + steam_cat].copy(), steam.hit.astype(int)
Xr, yr = sales[sales_num + sales_cat].copy(), sales.global_sales.astype(float)
groups = sales.name.str.normalize("NFKC").str.strip().str.casefold()
ic_train, ic_test = train_test_split(np.arange(len(Xc)), test_size=0.2,
                                     stratify=yc, random_state=SEED)
ir_train, ir_test = next(GroupShuffleSplit(n_splits=1, test_size=0.2,
                            random_state=SEED).split(Xr, yr, groups))
Xc_train, Xc_test, yc_train, yc_test = Xc.iloc[ic_train], Xc.iloc[ic_test], yc.iloc[ic_train], yc.iloc[ic_test]
Xr_train, Xr_test, yr_train, yr_test = Xr.iloc[ir_train], Xr.iloc[ir_test], yr.iloc[ir_train], yr.iloc[ir_test]
g_train, g_test = groups.iloc[ir_train], groups.iloc[ir_test]
assert not set(g_train) & set(g_test)
cv_c = list(StratifiedKFold(3, shuffle=True, random_state=SEED).split(Xc_train, yc_train))
cv_r = list(GroupKFold(3).split(Xr_train, yr_train, g_train))
for a, b in cv_r:
    assert not set(g_train.iloc[a]) & set(g_train.iloc[b])
display(pd.DataFrame({"Задача": ["Классификация", "Регрессия"],
    "train": [len(ic_train), len(ir_train)], "test": [len(ic_test), len(ir_test)]}))
print("Доля hit в train/test:", round(yc_train.mean(), 3), round(yc_test.mean(), 3))
np.savez(RESULTS / "split_indices.npz", ic_train=ic_train, ic_test=ic_test,
         ir_train=ir_train, ir_test=ir_test)

### Разведочный анализ только обучающей части
Смотрим дисбаланс, тяжёлый хвост продаж и масштабы дорелизных показателей. Эти графики мотивируют логарифмирование и устойчивое масштабирование; тестовые распределения для выбора решений не используются.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
yc_train.value_counts().sort_index().plot.bar(ax=axes[0, 0], rot=0, title="Steam: число игр по классам (train)")
sns.histplot(yr_train, bins=60, ax=axes[0, 1]); axes[0, 1].set_title("Продажи, млн копий (train)")
sns.histplot(np.log1p(yr_train), bins=60, ax=axes[1, 0]); axes[1, 0].set_title("log(1 + продажи), train")
sns.boxplot(data=np.log1p(Xc_train[["team_size", "wishlists_k", "trailer_views_k"]]), ax=axes[1, 1])
axes[1, 1].set_title("Логарифмы разных по масштабу признаков")
plt.tight_layout(); plt.show()
display(Xc_train[steam_num].describe().T)
display(Xr_train[sales_cat].nunique().rename("Число категорий в train").to_frame())

## 2. Бейзлайн sklearn и оценка качества

KNN предсказывает по ближайшим играм: классификация — доля голосов за класс, регрессия — среднее продаж. В базовом варианте k=5, равные веса, евклидово расстояние. Числа заполняем медианой и стандартизуем, категории — наиболее частым значением и one-hot; неизвестные категории допускаются. Даже простой KNN требует масштабирующей обработки. Dummy-модели дают минимальный ориентир.

Следующая ячейка определяет общие функции оценки. Таблица собирается по всем вариантам, а предсказания сохраняются для воспроизводимости.

In [ ]:
def preprocessor(num, cat, robust=False, rare=False):
    return ColumnTransformer([
        ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                          ("scale", RobustScaler() if robust else StandardScaler())]), num),
        ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                          ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False,
                               min_frequency=0.01 if rare else None))]), cat)
    ], sparse_threshold=0)

def baseline(task):
    num, cat = (steam_num, steam_cat) if task == "c" else (sales_num, sales_cat)
    model = KNeighborsClassifier() if task == "c" else KNeighborsRegressor()
    return Pipeline([("prep", preprocessor(num, cat)), ("knn", model)])

rows, predictions = [], {}
def evaluate(name, task, model):
    X, y = (Xc_test, yc_test) if task == "c" else (Xr_test, yr_test)
    start = time.perf_counter()
    if task == "c":
        prob = model.predict_proba(X)[:, 1]
        pred = (prob > 0.5).astype(int)
        metrics = {"AP": average_precision_score(y, prob), "ROC_AUC": roc_auc_score(y, prob),
            "F1": f1_score(y, pred, zero_division=0), "Precision": precision_score(y, pred, zero_division=0),
            "Recall": recall_score(y, pred, zero_division=0), "Balanced_accuracy": balanced_accuracy_score(y, pred),
            "Accuracy": accuracy_score(y, pred)}
        predictions[(task, name)] = (pred, prob)
    else:
        pred = model.predict(X)
        metrics = {"MAE": mean_absolute_error(y, pred), "RMSE": np.sqrt(mean_squared_error(y, pred)),
                   "R2": r2_score(y, pred)}
        predictions[(task, name)] = pred
    rows.append({"task": task, "model": name, **metrics, "predict_seconds": time.perf_counter() - start})
    return pd.DataFrame([rows[-1]])

base_c, base_r = baseline("c"), baseline("r")
base_c.fit(Xc_train, yc_train)
base_r.fit(Xr_train, yr_train)
dummy_c = DummyClassifier(strategy="prior").fit(Xc_train, yc_train)
dummy_r = DummyRegressor(strategy="median").fit(Xr_train, yr_train)
for name, task, model in [("Dummy", "c", dummy_c), ("Dummy", "r", dummy_r),
                          ("sklearn baseline", "c", base_c), ("sklearn baseline", "r", base_r)]:
    display(evaluate(name, task, model))

## 3. Улучшение бейзлайна

### 3a–3b. Проверяемые гипотезы
1. **H1:** RobustScaler и объединение редких категорий (частота <1% внутри обучающего fold) уменьшат влияние выбросов и редких издателей.
2. **H2:** log1p правосторонних числовых признаков Steam и производные признаки улучшат смысл расстояний. Добавляем признак бесплатной игры и log-отношение wishlist/просмотров. Для продаж проверяем взаимодействие платформа–жанр и десятилетие; они известны до продаж.
3. **H3:** подбор k, весов и нормы расстояния на CV улучшит результат. Для регрессии дополнительно проверяем log1p цели с обратным expm1. Такая модель оценивает другой центр распределения и может улучшить MAE, но ухудшить RMSE; это проверяется, а не предполагается.

H1 и H2 проверяем поэтапно при неизменных k=5. H3 — отдельный поиск параметров после H2. В окончательном выборе участвуют **все** кандидаты, включая исходный бейзлайн. Улучшение не гарантировано. Если CV не подтверждает гипотезу, её не объявляем успешной. Среднее ± стандартное отклонение CV описывает разброс по fold, а не доверительный интервал. Наивная оптимизация accuracy и подбор по тесту не используются.

### Детерминированное формирование признаков
Преобразователь совместим с Pipeline. Он не вычисляет статистик по всей таблице и не использует целевую переменную. Параметры последующих scaler и one-hot обучаются отдельно на каждом fold.

In [ ]:
class GameFeatures(TransformerMixin, BaseEstimator):
    def __init__(self, task="c"):
        self.task = task
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        out = X.copy()
        if self.task == "c":
            out["is_free"] = (out.price_usd == 0).astype(float)
            out["wishlist_trailer_ratio"] = np.log1p(out.wishlists_k) - np.log1p(out.trailer_views_k)
            for col in ["team_size", "price_usd", "file_size_gb", "wishlists_k", "trailer_views_k"]:
                out[col] = np.log1p(out[col])
        else:
            out["platform_genre"] = out.platform.fillna("unknown") + " / " + out.genre.fillna("unknown")
            out["decade"] = ((out.year // 10) * 10).astype("Int64").astype("string").fillna("unknown").astype(object)
        return out

def candidate(task, robust=False, rare=False, features=False):
    num = list(steam_num if task == "c" else sales_num)
    cat = list(steam_cat if task == "c" else sales_cat)
    steps = []
    if features:
        steps.append(("features", GameFeatures(task)))
        if task == "c": num += ["is_free", "wishlist_trailer_ratio"]
        else: cat += ["platform_genre", "decade"]
    steps += [("prep", preprocessor(num, cat, robust, rare)),
              ("knn", KNeighborsClassifier(algorithm="brute") if task == "c"
               else KNeighborsRegressor(algorithm="brute"))]
    return Pipeline(steps)

experiments, candidates = [], {"c": {}, "r": {}}
for task, X, y, folds, scoring in [("c", Xc_train, yc_train, cv_c, "average_precision"),
                                  ("r", Xr_train, yr_train, cv_r, "neg_mean_absolute_error")]:
    for name, model in [("H0 baseline", baseline(task)),
                         ("H1 robust + rare", candidate(task, True, True)),
                         ("H2 H1 + features/logX", candidate(task, True, True, True))]:
        scores = cross_val_score(model, X, y, cv=folds, scoring=scoring, n_jobs=2)
        candidates[task][name] = model
        experiments.append({"task": task, "experiment": name,
                            "selection_score": scores.mean(), "std": scores.std()})
cv_table = pd.DataFrame(experiments)
display(cv_table)

### Подбор гиперпараметров на тех же CV-разбиениях
Проверяем k ∈ {5, 15, 35}, равные/обратные расстоянию веса и p ∈ {1, 2}. В регрессии выполняем два одинаковых поиска: по исходной и по логарифмической цели. Препроцессинг остаётся внутри fold; оценка log-модели автоматически возвращается в миллионы копий.

In [ ]:
param_grid = {"knn__n_neighbors": [5, 15, 35], "knn__weights": ["uniform", "distance"], "knn__p": [1, 2]}
searches = {}
for task, X, y, folds, scoring in [("c", Xc_train, yc_train, cv_c, "average_precision"),
                                  ("r", Xr_train, yr_train, cv_r, "neg_mean_absolute_error")]:
    variants = [("H3 tuned", candidate(task, True, True, True), param_grid)]
    if task == "r":
        log_model = TransformedTargetRegressor(regressor=candidate(task, True, True, True),
                                              func=np.log1p, inverse_func=np.expm1)
        variants.append(("H3 tuned + logY", log_model,
                         {"regressor__" + k: v for k, v in param_grid.items()}))
    for name, model, grid in variants:
        search = GridSearchCV(model, grid, cv=folds, scoring=scoring,
                              n_jobs=2, refit=True, error_score="raise")
        search.fit(X, y)
        searches[(task, name)] = search
        candidates[task][name] = search.best_estimator_
        idx = search.best_index_
        experiments.append({"task": task, "experiment": name,
            "selection_score": search.best_score_, "std": search.cv_results_["std_test_score"][idx]})
        print(task, name, search.best_params_)
        pd.DataFrame(search.cv_results_).to_csv(RESULTS / f"grid_{task}_{'log' if 'logY' in name else 'raw'}.csv", index=False)
cv_table = pd.DataFrame(experiments)
display(cv_table)
cv_table.to_csv(RESULTS / "cv_experiments.csv", index=False)

### 3c–3g. Выбор улучшенного бейзлайна, обучение и сравнение
Сначала фиксируем победителя по CV, обучаем на всём train, затем вычисляем тестовые метрики. Название «улучшенный» обозначает процедуру выбора; фактический выигрыш на тесте проверяем в выводе. Повторный поиск по тесту не выполняется.

In [ ]:
selected, improved = {}, {}
for task, X, y in [("c", Xc_train, yc_train), ("r", Xr_train, yr_train)]:
    ranking = cv_table.loc[cv_table.task == task].sort_values("selection_score", ascending=False)
    name = ranking.iloc[0].experiment
    selected[task] = name
    improved[task] = clone(candidates[task][name]).fit(X, y)
    print("Выбрано по CV:", task, name)
    display(evaluate("sklearn improved", task, improved[task]))
    base_score = cv_table.query("task == @task and experiment == 'H0 baseline'").selection_score.iloc[0]
    gain = ranking.iloc[0].selection_score - base_score
    display(Markdown(f"**Вывод CV ({task}):** выбран `{name}`, выигрыш основной CV-оценки {gain:.4f}. "
                     "Для регрессии это уменьшение MAE; для классификации — увеличение AP."))
    ordered = cv_table[cv_table.task == task].reset_index(drop=True)
    for prev, cur in [(0, 1), (1, 2)]:
        delta = ordered.iloc[cur].selection_score - ordered.iloc[prev].selection_score
        print(ordered.iloc[cur].experiment, "по сравнению с предыдущим этапом:", round(delta, 5),
              "— поддержана по среднему CV" if delta > 0 else "— не поддержана по среднему CV")
comparison = pd.DataFrame(rows)
for task, metric in [("c", "AP"), ("r", "MAE")]:
    table = comparison[comparison.task == task]
    a = table.loc[table.model == "sklearn baseline", metric].iloc[0]
    b = table.loc[table.model == "sklearn improved", metric].iloc[0]
    display(Markdown(f"**Тест ({task}):** {metric}: {a:.4f} → {b:.4f}; изменение {b-a:+.4f}. "
                    + ("Качество выросло." if (b > a if task == "c" else b < a) else "Улучшение на тесте не подтверждено.")))

## 4. Собственная реализация KNN

### 4a. Алгоритм
Для каждого запроса вычисляем расстояние Минковского (p=1: сумма модулей; p=2: корень из суммы квадратов), выбираем k ближайших строк и агрегируем их ответы. При `uniform` веса равны 1, при `distance` — 1/d. Если есть точные совпадения, голосуют только они; это соответствует правилу sklearn. В классификации вероятность — взвешенная доля голосов, при равенстве побеждает меньший класс. В регрессии — взвешенное среднее.

Соседи выбираются через NumPy; sklearn не используется для поиска, расстояний и агрегации. Используются только интерфейсы estimator и проверки входа sklearn. Пакетная обработка ограничивает память: расстояния имеют размер batch × train, без трёхмерного массива batch × train × features. Для p=2 используем тождество ‖x−z‖² = ‖x‖² + ‖z‖² − 2x·z и матричное умножение NumPy. Почти нулевые расстояния пересчитываем напрямую, чтобы избежать ошибки округления на точных совпадениях. Вычисление расстояний требует O(n_test × n_train × d), полная стабильная сортировка — ещё O(n_test × n_train × log(n_train)); обучение хранит данные, память O(n_train × d). Сортировка при равных расстояниях стабильная: меньший индекс обучающей строки идёт первым. sklearn может выбрать иной набор соседей на границе k при равных расстояниях, что особенно вероятно у категориальных признаков продаж.

### Базовый класс и две модели
Проверяем гиперпараметры и размерности, поддерживаем `clone` и Pipeline. Реализуем `predict_proba` для AP и ROC AUC.

In [ ]:
class NumpyKNNBase(BaseEstimator):
    def __init__(self, n_neighbors=5, weights="uniform", p=2, batch_size=64):
        self.n_neighbors = n_neighbors
        self.weights = weights
        self.p = p
        self.batch_size = batch_size
    def fit(self, X, y):
        X, y = check_X_y(X, y, dtype=float)
        if not isinstance(self.n_neighbors, (int, np.integer)) or not 1 <= self.n_neighbors <= len(X):
            raise ValueError("n_neighbors должен быть целым от 1 до размера train")
        if self.weights not in ("uniform", "distance") or self.p not in (1, 2):
            raise ValueError("Поддерживаются uniform/distance и p=1/2")
        if not isinstance(self.batch_size, (int, np.integer)) or self.batch_size < 1:
            raise ValueError("batch_size должен быть положительным целым")
        self.X_ = np.array(X, copy=True, order="F")
        self.y_ = np.array(y, copy=True)
        self.n_features_in_ = X.shape[1]
        return self
    def _neighbors(self, X):
        check_is_fitted(self, ["X_", "y_"])
        X = check_array(X, dtype=float)
        if X.shape[1] != self.n_features_in_:
            raise ValueError("Число признаков отличается от train")
        for start in range(0, len(X), self.batch_size):
            batch = X[start:start + self.batch_size]
            if self.p == 2:
                norm_q = np.einsum("ij,ij->i", batch, batch)
                norm_t = np.einsum("ij,ij->i", self.X_, self.X_)
                distances = norm_q[:, None] + norm_t[None, :] - 2 * (batch @ self.X_.T)
                tolerance = 64 * np.finfo(float).eps * np.maximum(1, norm_q[:, None] + norm_t[None, :])
                for row in range(len(batch)):
                    near = np.flatnonzero(distances[row] <= tolerance[row])
                    diff = self.X_[near] - batch[row]
                    distances[row, near] = np.einsum("ij,ij->i", diff, diff)
                np.sqrt(np.maximum(distances, 0), out=distances)
            else:
                distances = np.zeros((len(batch), len(self.X_)), dtype=float)
                for j in range(self.n_features_in_):
                    distances += np.abs(batch[:, j, None] - self.X_[None, :, j])
            indices = np.argsort(distances, axis=1, kind="stable")[:, :self.n_neighbors]
            dist = np.take_along_axis(distances, indices, axis=1)
            if self.weights == "uniform":
                weights = np.ones_like(dist)
            else:
                zero = dist == 0
                weights = 1 / np.maximum(dist, np.finfo(float).tiny)
                has_zero = zero.any(axis=1)
                weights[has_zero] = zero[has_zero].astype(float)
            weights /= weights.sum(axis=1, keepdims=True)
            yield indices, weights

class NumpyKNNClassifier(ClassifierMixin, NumpyKNNBase):
    def fit(self, X, y):
        super().fit(X, y)
        self.classes_, self.encoded_y_ = np.unique(self.y_, return_inverse=True)
        return self
    def predict_proba(self, X):
        result = []
        for indices, weights in self._neighbors(X):
            labels = self.encoded_y_[indices]
            result.append(np.column_stack([
                (weights * (labels == c)).sum(axis=1) for c in range(len(self.classes_))]))
        return np.vstack(result)
    def predict(self, X):
        return self.classes_[self.predict_proba(X).argmax(axis=1)]

class NumpyKNNRegressor(RegressorMixin, NumpyKNNBase):
    def predict(self, X):
        return np.concatenate([(weights * self.y_[indices]).sum(axis=1)
                               for indices, weights in self._neighbors(X)])

### Проверка корректности на контролируемых примерах
Проверяем обе нормы, оба типа весов, вероятности, точные совпадения с несколькими ответами, ничью классов, пакетную обработку и отклонение неверного k. На случайных непрерывных данных без равных расстояний требуем совпадения с sklearn до численной точности. Это отдельные проверки алгоритма, а не подгонка под тестовые данные.

In [ ]:
rng = np.random.default_rng(SEED)
small_X, query = rng.normal(size=(40, 4)), rng.normal(size=(9, 4))
small_c, small_r = rng.integers(0, 3, 40), rng.normal(size=40)
for p in (1, 2):
    for weights in ("uniform", "distance"):
        params = dict(n_neighbors=7, p=p, weights=weights)
        own_c = NumpyKNNClassifier(**params, batch_size=3).fit(small_X, small_c)
        ref_c = KNeighborsClassifier(**params, algorithm="brute").fit(small_X, small_c)
        own_r = NumpyKNNRegressor(**params, batch_size=3).fit(small_X, small_r)
        ref_r = KNeighborsRegressor(**params, algorithm="brute").fit(small_X, small_r)
        np.testing.assert_allclose(own_c.predict_proba(query), ref_c.predict_proba(query), atol=1e-12)
        np.testing.assert_allclose(own_r.predict(query), ref_r.predict(query), atol=1e-12)
        np.testing.assert_array_equal(own_c.predict(query), ref_c.predict(query))
for p in (1, 2):
    own = NumpyKNNRegressor(3, "distance", p).fit([[0], [0], [2]], [2., 4., 99.])
    np.testing.assert_allclose(own.predict([[0]]), [3.])
    own = NumpyKNNClassifier(3, "distance", p).fit([[0], [0], [2]], [0, 1, 1])
    np.testing.assert_allclose(own.predict_proba([[0]]), [[0.5, 0.5]])
    np.testing.assert_array_equal(own.predict([[0]]), [0])
try:
    NumpyKNNRegressor(n_neighbors=4).fit([[0], [1]], [0., 1.])
except ValueError:
    pass
else:
    raise AssertionError("Неверный k не был отклонён")
print("Все проверки собственной реализации пройдены.")

### 4b–4e. Собственные модели с исходным бейзлайном
Копируем структуру и параметры sklearn-бейзлайна, заменяем только алгоритм соседей. Учим преобразователи и собственную модель на том же train. Так сравнение проверяет реализацию, а не разницу в подготовке данных.

In [ ]:
def replace_knn(model, task):
    result = clone(model)
    if isinstance(result, TransformedTargetRegressor):
        result.regressor = replace_knn(result.regressor, task)
    else:
        params = result.named_steps["knn"].get_params()
        cls = NumpyKNNClassifier if task == "c" else NumpyKNNRegressor
        result.set_params(knn=cls(**{k: params[k] for k in ["n_neighbors", "weights", "p"]}))
    return result
own_base_c = replace_knn(base_c, "c").fit(Xc_train, yc_train)
own_base_r = replace_knn(base_r, "r").fit(Xr_train, yr_train)
display(evaluate("numpy baseline", "c", own_base_c))
display(evaluate("numpy baseline", "r", own_base_r))
for task, metric in [("c", "AP"), ("r", "MAE")]:
    table = pd.DataFrame(rows).query("task == @task").set_index("model")
    delta = table.loc["numpy baseline", metric] - table.loc["sklearn baseline", metric]
    display(Markdown(f"**4e, {task}:** собственный baseline минус sklearn baseline по {metric}: {delta:+.6f}. "
                     "Близость результатов проверяет корректность; расхождения при равных расстояниях разбираются ниже."))

### 4f–4j. Перенос улучшений в собственную реализацию
Берём **выбранные по train-CV** признаки, scaler, кодировщик, k, веса, расстояние и, если выбран, log1p цели. Новый подбор по тесту не выполняем. Обучаем на том же train и сравниваем со sklearn из пункта 3.

In [ ]:
own_improved = {}
for task, X, y in [("c", Xc_train, yc_train), ("r", Xr_train, yr_train)]:
    own_improved[task] = replace_knn(improved[task], task).fit(X, y)
    display(evaluate("numpy improved", task, own_improved[task]))

### Итоговые таблицы и численное сравнение реализаций
Сопоставляем базовые и улучшенные модели, сохраняем метрики и предсказания. Проверка на полных данных диагностическая: при совпадающих расстояниях на границе k обе реализации могут корректно выбрать разных соседей. Строгая проверка без ничьих выполнена выше.

Для регрессии дополнительно проверяем все существенно различающиеся прогнозы: сравниваем упорядоченные расстояния до выбранных соседей обеих моделей. Совпадение расстояний при разных ответах показывает влияние ничьих и округления при выборе соседей.

In [ ]:
metrics = pd.DataFrame(rows)
display(metrics.loc[metrics.task == "c", ["model", "AP", "F1", "Precision", "Recall", "Balanced_accuracy", "ROC_AUC", "Accuracy", "predict_seconds"]])
display(metrics.loc[metrics.task == "r", ["model", "MAE", "RMSE", "R2", "predict_seconds"]])
metrics.to_csv(RESULTS / "test_metrics.csv", index=False)
for task in ["c", "r"]:
    for stage in ["baseline", "improved"]:
        ref, own = predictions[(task, "sklearn " + stage)], predictions[(task, "numpy " + stage)]
        if task == "c":
            diff = np.abs(ref[1] - own[1])
            print(task, stage, "Макс. отличие вероятностей:", diff.max(),
                  "Доля одинаковых классов:", np.mean(ref[0] == own[0]))
        else:
            diff = np.abs(ref - own)
            print(task, stage, "Макс. отличие продаж:", diff.max(),
                  "Среднее отличие:", diff.mean(), "Доля близких:", np.mean(np.isclose(ref, own)))
for task, y in [("c", yc_test), ("r", yr_test)]:
    output = pd.DataFrame({"row_index": y.index, "target": y.to_numpy()})
    for (t, name), pred in predictions.items():
        if t == task:
            output[name] = pred[0] if task == "c" else pred
            if task == "c": output[name + " probability"] = pred[1]
    output.to_csv(RESULTS / f"predictions_{task}.csv", index=False)

tie_rows = []
for stage, ref_model, own_model in [("baseline", base_r, own_base_r),
                                     ("improved", improved["r"], own_improved["r"])]:
    ref_pipe = ref_model.regressor_ if isinstance(ref_model, TransformedTargetRegressor) else ref_model
    own_pipe = own_model.regressor_ if isinstance(own_model, TransformedTargetRegressor) else own_model
    own_knn = own_pipe.named_steps["knn"]
    query = own_pipe[:-1].transform(Xr_test)
    np.testing.assert_allclose(ref_pipe[:-1].transform(Xr_train), own_knn.X_, atol=1e-12)
    different = np.flatnonzero(~np.isclose(predictions[("r", "sklearn " + stage)],
                                          predictions[("r", "numpy " + stage)], rtol=1e-8, atol=1e-10))
    explained, offset = 0, 0
    if len(different):
        ref_indices = ref_pipe.named_steps["knn"].kneighbors(query[different], return_distance=False)
        for own_indices, _ in own_knn._neighbors(query[different]):
            count = len(own_indices)
            batch = query[different[offset:offset+count]]
            ref_ids = ref_indices[offset:offset+count]
            d_own = np.linalg.norm(own_knn.X_[own_indices] - batch[:, None, :], ord=own_knn.p, axis=2)
            d_ref = np.linalg.norm(own_knn.X_[ref_ids] - batch[:, None, :], ord=own_knn.p, axis=2)
            explained += np.isclose(np.sort(d_own, axis=1), np.sort(d_ref, axis=1),
                                    rtol=1e-8, atol=1e-10).all(axis=1).sum()
            offset += count
    tie_rows.append({"stage": stage, "different_predictions": len(different),
                     "same_neighbor_distances": int(explained)})
tie_diagnostics = pd.DataFrame(tie_rows)
display(tie_diagnostics)
tie_diagnostics.to_csv(RESULTS / "neighbor_tie_diagnostics.csv", index=False)
print("Одинаковые наборы расстояний при разных прогнозах подтверждают влияние ничьих и округления при выборе соседей.")


### Визуализация качества
PR-кривые показывают компромисс precision/recall, матрица ошибок — поведение фиксированного порога. Для регрессии отображаем логарифмические координаты фактических/предсказанных продаж и остатки в исходных единицах. Графики служат анализом результатов, без последующей настройки по тесту.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 10))
for stage in ["baseline", "improved"]:
    PrecisionRecallDisplay.from_predictions(yc_test, predictions[("c", "sklearn " + stage)][1],
                                            name=stage, ax=axes[0, 0])
axes[0, 0].axhline(yc_test.mean(), linestyle="--", color="gray", label="Доля hit")
axes[0, 0].legend()
ConfusionMatrixDisplay.from_predictions(yc_test, predictions[("c", "sklearn improved")][0],
                                        ax=axes[0, 1], colorbar=False)
r_pred = predictions[("r", "sklearn improved")]
axes[1, 0].scatter(np.log1p(yr_test), np.log1p(r_pred), alpha=0.3, s=12)
limit = float(np.log1p(yr_test).max())
axes[1, 0].plot([0, limit], [0, limit], "r--")
axes[1, 0].set(xlabel="log1p фактических продаж", ylabel="log1p прогноза")
axes[1, 1].scatter(r_pred, yr_test.to_numpy() - r_pred, alpha=0.3, s=12)
axes[1, 1].axhline(0, color="red", linestyle="--")
axes[1, 1].set(xlabel="Прогноз, млн копий", ylabel="Факт − прогноз, млн копий")
plt.tight_layout(); plt.show()

### Итоговые выводы по фактическим результатам
Следующая ячейка формирует выводы автоматически: показывает результат выбора, выигрыш/проигрыш на тесте и различие собственной реализации со sklearn.

In [ ]:
for task, metric, label in [("c", "AP", "Классификация"), ("r", "MAE", "Регрессия")]:
    tab = metrics[metrics.task == task].set_index("model")
    a, b = tab.loc["sklearn baseline", metric], tab.loc["sklearn improved", metric]
    own_a, own_b = tab.loc["numpy baseline", metric], tab.loc["numpy improved", metric]
    benefit = b - a if task == "c" else a - b
    display(Markdown(f"**{label}.** По CV выбран `{selected[task]}`. "
        f"На тесте sklearn: {metric} {a:.4f} → {b:.4f}. "
        f"Выигрыш в нужном направлении: {benefit:+.4f}. "
        f"Собственная модель: {own_a:.4f} → {own_b:.4f}; "
        f"разница с улучшенной sklearn: {own_b-b:+.6f}. "
        + ("Перенос улучшений подтвердился на тесте." if benefit > 0
           else "CV-выбор не дал выигрыша на этом тесте; параметры после этого не меняем.")))
reg_table = metrics[metrics.task == "r"].set_index("model")
dummy_mae = reg_table.loc["Dummy", "MAE"]
for name in ["sklearn improved", "numpy improved"]:
    value = reg_table.loc[name, "MAE"]
    display(Markdown(f"**Сравнение с Dummy ({name}):** MAE {value:.6f}, "
        f"у медианного прогноза {dummy_mae:.6f} млн копий. "
        + ("Модель лучше простого ориентира по MAE." if value < dummy_mae else
           "Модель не превзошла простой ориентир по MAE; уменьшение ошибки относительно исходного KNN ещё не означает практической полезности.")))